# Q3 — Text Summarization (CNN/DailyMail)

**Canonical path**: Capped TextRank vs DistilBART comparison (20 train / 20 val / 20 test).

**Runtime**: T4 GPU required for BART cell.

> This notebook reproduces the report-facing Q3 comparison.
> For larger budgets or Lead-3, see the **Exploratory** section at the bottom.

## 1. Environment Setup

In [ ]:
import os, sys, json, glob, shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ON_COLAB = True
except ImportError:
    ON_COLAB = False
    print('Not running on Colab — skipping Drive mount.')

DRIVE_OUTPUT = '/content/drive/MyDrive/467-takehome-outputs/q3' if ON_COLAB else None
if DRIVE_OUTPUT:
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
REPO_DIR = '/content/467-takehome' if ON_COLAB else os.path.abspath('..')
if ON_COLAB:
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/zubeyralmaho/467-takehome.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
def save_to_drive(run_dir, tag=''):
    if not DRIVE_OUTPUT:
        return
    name = os.path.basename(run_dir) + (f'_{tag}' if tag else '')
    dest = os.path.join(DRIVE_OUTPUT, name)
    shutil.copytree(run_dir, dest, dirs_exist_ok=True)
    print(f'Saved to Drive: {dest}')

def gpu_cleanup():
    import gc
    torch.cuda.empty_cache()
    gc.collect()
    if torch.cuda.is_available():
        print(f'GPU memory freed. Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## 2. Canonical Run — Capped TextRank vs DistilBART

Matches the report comparison: 20 train / 20 val / 20 test, TextRank + BART on the same samples.

### 2a. TextRank (CPU)

In [ ]:
!python -m src.q3_summarization.main \
    --config configs/q3.yaml \
    --final-eval \
    --override \
        dataset.limit_train_samples=20 \
        dataset.limit_validation_samples=20 \
        dataset.limit_test_samples=20 \
        models.textrank.enabled=true \
        models.lead3.enabled=false \
        models.bart.enabled=false

In [ ]:
textrank_run = sorted(glob.glob('outputs/q3/run_*'))[-1]
save_to_drive(textrank_run, 'textrank')
print(f'TextRank run: {textrank_run}')

### 2b. DistilBART (GPU)

In [ ]:
!python -m src.q3_summarization.main \
    --config configs/q3.yaml \
    --final-eval \
    --override \
        dataset.limit_train_samples=20 \
        dataset.limit_validation_samples=20 \
        dataset.limit_test_samples=20 \
        models.textrank.enabled=false \
        models.lead3.enabled=false \
        models.bart.enabled=true \
        models.bart.model_name=sshleifer/distilbart-cnn-12-6 \
        models.bart.batch_size=2 \
        models.bart.max_input_length=512 \
        models.bart.max_output_length=80 \
        models.bart.min_output_length=20

In [ ]:
bart_run = sorted(glob.glob('outputs/q3/run_*'))[-1]
save_to_drive(bart_run, 'bart')
gpu_cleanup()

## 3. Regenerate Report Figure

The report comparison figure is produced by `scripts/report_comparison_figures.py`.

In [ ]:
print('Canonical Q3 runs:')
print(f'  TextRank: {textrank_run}')
print(f'  BART:     {bart_run}')
print()
print('To regenerate the Q3 report figure:')
print('  python scripts/report_comparison_figures.py')
print('  (default reads from outputs/q3/run_20260413_192426 — update path if needed)')

## 4. Results Summary

In [ ]:
print('=== Q3 Canonical Results ===')
for label, run_dir in [('TextRank', textrank_run), ('DistilBART', bart_run)]:
    metrics_file = os.path.join(run_dir, 'metrics.json')
    if os.path.exists(metrics_file):
        with open(metrics_file) as f:
            m = json.load(f)
        print(f'\n--- {label} ({os.path.basename(run_dir)}) ---')
        print(json.dumps(m, indent=2, default=str)[:2000])
    else:
        print(f'\n--- {label}: metrics.json not found ---')

---
## (Exploratory) Larger Budget / Lead-3 Runs

These runs are **not** part of the canonical report comparison. Uncomment to run.

In [ ]:
# # --- Lead-3 + TextRank on 1K samples ---
# !python -m src.q3_summarization.main \
#     --config configs/q3.yaml \
#     --final-eval \
#     --override \
#         dataset.limit_train_samples=1000 \
#         dataset.limit_validation_samples=1000 \
#         dataset.limit_test_samples=1000 \
#         models.textrank.enabled=true \
#         models.lead3.enabled=true \
#         models.bart.enabled=false

# # --- BART on 1K samples ---
# !python -m src.q3_summarization.main \
#     --config configs/q3.yaml \
#     --final-eval \
#     --override \
#         dataset.limit_train_samples=1000 \
#         dataset.limit_validation_samples=1000 \
#         dataset.limit_test_samples=1000 \
#         models.textrank.enabled=false \
#         models.lead3.enabled=false \
#         models.bart.enabled=true \
#         models.bart.batch_size=4 \
#         models.bart.max_input_length=1024